# Cost & Latency Optimization — Making Agents Efficient

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/27_cost_and_latency_optimization.ipynb)

## What This Notebook Teaches You

A single agent task can involve **5–20 LLM calls**. At scale, this means thousands of tokens per request and real dollar costs. This notebook builds practical optimization techniques:

1. **Token budget tracking** — monitor and limit token consumption per task
2. **Response caching** — hash prompts and reuse identical responses
3. **Prompt compression** — reduce context size without losing information
4. **Model routing** — use cheap settings for simple tasks, expensive settings for hard ones
5. **Early termination** — detect circular reasoning and stop wasting tokens
6. **Comparative experiment** — measure savings across all optimizations

Each technique is implemented as a reusable class that wraps the `generate()` function.

---

> **Prerequisites**: Module 26 (Agent Evaluation & Testing) for the evaluation framework.
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The Cost Problem

### Why Agent Costs Explode

| Pattern | LLM Calls | Why |
|---------|-----------|-----|
| Single prompt | 1 | One question, one answer |
| Chain of prompts | 3–5 | Sequential steps |
| ReAct agent | 5–15 | Reason → Act → Observe loop |
| Multi-agent system | 20–50+ | Multiple agents, each with their own loops |

Each call includes the **full conversation history** as input tokens. A 5-step ReAct loop with growing context can easily consume **10,000+ tokens** for a single task.

### The Optimization Toolkit

We'll build five optimization layers that compose together:

```
User Query
  → TokenBudget (track & limit)
    → ResponseCache (reuse identical calls)
      → PromptCompressor (shrink context)
        → ModelRouter (cheap vs expensive settings)
          → CircularityDetector (stop loops)
            → LLM
```

## 2. Token Budget Tracker

The first step to optimization is **measurement**. Our TokenBudget class wraps every LLM call and tracks token usage with alerts at configurable thresholds.

In [ ]:
class TokenBudget:
    """Track token consumption with budget limits and alerts."""

    def __init__(self, max_tokens: int = 50000, alert_thresholds: List[float] = None):
        """
        Args:
            max_tokens: Maximum total tokens allowed for the task.
            alert_thresholds: Fractions at which to alert (e.g., [0.5, 0.8, 0.95]).
        """
        self.max_tokens = max_tokens
        self.alert_thresholds = alert_thresholds or [0.5, 0.8, 0.95]
        self.alerts_fired: set = set()
        self.calls: List[Dict] = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0

    def estimate_tokens(self, text: str) -> int:
        """Rough token estimate: ~1.3 tokens per word for English."""
        return int(len(text.split()) * 1.3)

    @property
    def total_tokens(self) -> int:
        return self.total_input_tokens + self.total_output_tokens

    @property
    def budget_remaining(self) -> int:
        return max(0, self.max_tokens - self.total_tokens)

    @property
    def utilization(self) -> float:
        return self.total_tokens / self.max_tokens if self.max_tokens > 0 else 0

    def record_call(self, input_text: str, output_text: str, label: str = ""):
        """Record a single LLM call's token usage."""
        input_tokens = self.estimate_tokens(input_text)
        output_tokens = self.estimate_tokens(output_text)
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens

        self.calls.append({
            "label": label,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cumulative": self.total_tokens,
            "utilization": self.utilization,
        })

        # Check thresholds
        for threshold in self.alert_thresholds:
            if self.utilization >= threshold and threshold not in self.alerts_fired:
                self.alerts_fired.add(threshold)
                print(f"  ⚠ Budget alert: {threshold:.0%} used "
                      f"({self.total_tokens:,}/{self.max_tokens:,} tokens)")

    def is_over_budget(self) -> bool:
        return self.total_tokens >= self.max_tokens

    def summary(self) -> str:
        return (
            f"Budget: {self.total_tokens:,}/{self.max_tokens:,} tokens "
            f"({self.utilization:.1%}) | "
            f"Calls: {len(self.calls)} | "
            f"Input: {self.total_input_tokens:,} | Output: {self.total_output_tokens:,}"
        )

# Demo: Simulate tracking across multiple calls
budget = TokenBudget(max_tokens=5000)

# Simulate 5 agent steps with growing context
for i in range(5):
    fake_input = "What is the meaning of life? " * (10 * (i + 1))  # growing context
    fake_output = "The answer involves many considerations. " * 5
    budget.record_call(fake_input, fake_output, label=f"step-{i+1}")

print(f"\nFinal: {budget.summary()}")
print(f"Over budget? {budget.is_over_budget()}")
print(f"\nCall-by-call breakdown:")
for call in budget.calls:
    print(f"  {call['label']}: +{call['input_tokens']+call['output_tokens']} tokens "
          f"(cumulative: {call['cumulative']}, {call['utilization']:.1%})")

## 3. Response Caching

Many agent tasks produce **identical or near-identical prompts** across runs. Caching responses avoids redundant LLM calls entirely.

Our approach:
- Hash the full prompt (messages + generation params) to a cache key
- Store responses in a dictionary
- On cache hit: return immediately (zero tokens, zero latency)
- Track cache hit rates for monitoring

In [ ]:
import hashlib

class ResponseCache:
    """Cache LLM responses by prompt hash."""

    def __init__(self, max_size: int = 1000):
        self.cache: Dict[str, str] = {}
        self.max_size = max_size
        self.hits = 0
        self.misses = 0

    def _make_key(self, messages: List[Dict], **kwargs) -> str:
        """Create a deterministic hash from messages and generation params."""
        content = json.dumps(messages, sort_keys=True)
        # Include generation params that affect output
        for k in sorted(kwargs.keys()):
            content += f"|{k}={kwargs[k]}"
        return hashlib.sha256(content.encode()).hexdigest()[:16]

    def get(self, messages: List[Dict], **kwargs) -> Optional[str]:
        """Look up cached response. Returns None on miss."""
        key = self._make_key(messages, **kwargs)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None

    def put(self, messages: List[Dict], response: str, **kwargs):
        """Store a response in the cache."""
        if len(self.cache) >= self.max_size:
            # Evict oldest entry (simple FIFO)
            oldest_key = next(iter(self.cache))
            del self.cache[oldest_key]
        key = self._make_key(messages, **kwargs)
        self.cache[key] = response

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0

    def stats(self) -> str:
        total = self.hits + self.misses
        return (
            f"Cache: {len(self.cache)} entries | "
            f"Hits: {self.hits}/{total} ({self.hit_rate:.1%}) | "
            f"Misses: {self.misses}"
        )

def generate_cached(messages, cache: ResponseCache, **kwargs) -> Tuple[str, bool]:
    """Generate with caching. Returns (response, was_cached)."""
    cached = cache.get(messages, **kwargs)
    if cached is not None:
        return cached, True

    response = generate(messages, **kwargs)
    cache.put(messages, response, **kwargs)
    return response, False

# Demo: Show caching in action
cache = ResponseCache()

test_messages = [{"role": "user", "content": "What is 2 + 2?"}]

print("Call 1 (cache miss):")
start = time.time()
resp1, was_cached = generate_cached(test_messages, cache, max_new_tokens=64, temperature=0.1)
t1 = time.time() - start
print(f"  Response: {resp1[:80]}...")
print(f"  Cached: {was_cached} | Time: {t1:.2f}s")

print("\nCall 2 (cache hit — same prompt):")
start = time.time()
resp2, was_cached = generate_cached(test_messages, cache, max_new_tokens=64, temperature=0.1)
t2 = time.time() - start
print(f"  Response: {resp2[:80]}...")
print(f"  Cached: {was_cached} | Time: {t2:.4f}s")

print(f"\nSpeedup: {t1/max(t2, 0.0001):.0f}x")
print(cache.stats())

## 4. Prompt Compression

As a ReAct agent runs, its conversation history grows with every step. By step 5, the model re-reads all previous thoughts, actions, and observations — most of which are no longer relevant.

**Prompt compression** summarizes the conversation history before sending it to the model, reducing input tokens dramatically.

In [ ]:
class PromptCompressor:
    """Compress conversation history to reduce token count."""

    def __init__(self, max_history_tokens: int = 1500, summary_trigger: int = 3):
        """
        Args:
            max_history_tokens: Compress when history exceeds this token estimate.
            summary_trigger: Number of assistant turns before attempting compression.
        """
        self.max_history_tokens = max_history_tokens
        self.summary_trigger = summary_trigger
        self.compressions = 0
        self.tokens_saved = 0

    def estimate_tokens(self, messages: List[Dict]) -> int:
        total_text = " ".join(m["content"] for m in messages)
        return int(len(total_text.split()) * 1.3)

    def compress(self, messages: List[Dict]) -> List[Dict]:
        """Compress conversation history if it's too long.

        Strategy:
        1. Keep the system message (first message) and last 2 messages intact
        2. Summarize everything in between into a compact context message
        """
        if len(messages) < self.summary_trigger + 2:
            return messages  # Too short to compress

        current_tokens = self.estimate_tokens(messages)
        if current_tokens <= self.max_history_tokens:
            return messages  # Under budget

        # Split: system + middle + recent
        system_msg = messages[0] if messages[0]["role"] == "system" else None
        start_idx = 1 if system_msg else 0
        middle = messages[start_idx:-2]
        recent = messages[-2:]

        if not middle:
            return messages  # Nothing to compress

        # Build summary of middle messages
        summary_parts = []
        for msg in middle:
            role = msg["role"]
            content = msg["content"]
            if role == "assistant":
                # Extract key actions from agent responses
                thought = re.search(r'Thought:\s*(.+?)(?:\n|$)', content)
                action = re.search(r'Action:\s*(.+?)(?:\n|$)', content)
                if thought and action:
                    summary_parts.append(f"Thought: {thought.group(1)[:80]} → Action: {action.group(1)}")
                elif content.strip():
                    summary_parts.append(f"Agent: {content[:100]}")
            elif role == "user" and content.startswith("Observation:"):
                summary_parts.append(content[:120])

        summary_text = "Previous conversation summary:\n" + "\n".join(summary_parts)

        compressed = []
        if system_msg:
            compressed.append(system_msg)
        compressed.append({"role": "user", "content": summary_text})
        compressed.extend(recent)

        new_tokens = self.estimate_tokens(compressed)
        saved = current_tokens - new_tokens
        if saved > 0:
            self.compressions += 1
            self.tokens_saved += saved

        return compressed

    def stats(self) -> str:
        return (
            f"Compressions: {self.compressions} | "
            f"Tokens saved: {self.tokens_saved:,}"
        )

# Demo: Show compression on a simulated conversation
compressor = PromptCompressor(max_history_tokens=200, summary_trigger=2)

long_conversation = [
    {"role": "system", "content": "You are a helpful assistant with tools."},
    {"role": "user", "content": "Calculate 15% tip on $84.50"},
    {"role": "assistant", "content": "Thought: I need to calculate 15% of 84.50\nAction: calculator\nAction Input: 84.50 * 0.15"},
    {"role": "user", "content": "Observation: 12.675"},
    {"role": "assistant", "content": "Thought: I need to round to nearest cent\nAction: calculator\nAction Input: round(12.675, 2)"},
    {"role": "user", "content": "Observation: 12.68"},
    {"role": "assistant", "content": "Thought: Now I have the answer.\nAction: calculator\nAction Input: 84.50 + 12.68"},
    {"role": "user", "content": "Observation: 97.18"},
]

original_tokens = compressor.estimate_tokens(long_conversation)
compressed = compressor.compress(long_conversation)
compressed_tokens = compressor.estimate_tokens(compressed)

print(f"Original: {len(long_conversation)} messages, ~{original_tokens} tokens")
print(f"Compressed: {len(compressed)} messages, ~{compressed_tokens} tokens")
print(f"Saved: ~{original_tokens - compressed_tokens} tokens ({(original_tokens - compressed_tokens)/original_tokens:.0%})")
print(f"\nCompressed messages:")
for msg in compressed:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")
print(f"\n{compressor.stats()}")

## 5. Model Routing — Smart Resource Allocation

Not every task needs the same compute budget. A simple factual lookup doesn't need the same token generation as a complex multi-step reasoning problem.

**Model routing** classifies task difficulty and adjusts generation parameters accordingly. Since we're using a single model (Qwen3-8B), we route between different generation settings rather than different models.

In [ ]:
class ModelRouter:
    """Route tasks to different generation configurations based on difficulty.

    Since we use a single open-source model, routing means adjusting:
    - max_new_tokens: how much output to generate
    - temperature: randomness (lower = more focused, higher = more creative)
    - do_sample: greedy vs sampling
    """

    PROFILES = {
        "simple": {
            "max_new_tokens": 128,
            "temperature": 0.1,
            "do_sample": True,
            "description": "Fast, focused — for factual lookups and simple math",
        },
        "moderate": {
            "max_new_tokens": 256,
            "temperature": 0.5,
            "do_sample": True,
            "description": "Balanced — for standard reasoning tasks",
        },
        "complex": {
            "max_new_tokens": 512,
            "temperature": 0.7,
            "do_sample": True,
            "description": "Full power — for multi-step reasoning and creative tasks",
        },
    }

    # Keywords that suggest different difficulty levels
    SIMPLE_SIGNALS = ["what is", "capital of", "symbol for", "how many", "boiling point"]
    COMPLEX_SIGNALS = ["if .* then", "compare", "explain why", "step by step",
                       "calculate .* and .* then", "after .* what"]

    def __init__(self):
        self.routing_log: List[Dict] = []

    def classify_difficulty(self, query: str) -> str:
        """Classify a query's difficulty based on linguistic signals."""
        query_lower = query.lower()

        # Check for complex signals
        for pattern in self.COMPLEX_SIGNALS:
            if re.search(pattern, query_lower):
                return "complex"

        # Check for simple signals
        for signal in self.SIMPLE_SIGNALS:
            if signal in query_lower:
                return "simple"

        return "moderate"  # default

    def get_params(self, query: str) -> Dict:
        """Get generation parameters for a query."""
        difficulty = self.classify_difficulty(query)
        profile = self.PROFILES[difficulty]

        self.routing_log.append({
            "query": query[:50],
            "difficulty": difficulty,
            "max_new_tokens": profile["max_new_tokens"],
        })

        return {
            "max_new_tokens": profile["max_new_tokens"],
            "temperature": profile["temperature"],
            "do_sample": profile["do_sample"],
        }

    def stats(self) -> str:
        if not self.routing_log:
            return "No queries routed yet."
        counts = defaultdict(int)
        for entry in self.routing_log:
            counts[entry["difficulty"]] += 1
        total_tokens_budget = sum(e["max_new_tokens"] for e in self.routing_log)
        max_possible = len(self.routing_log) * 512
        return (
            f"Routed {len(self.routing_log)} queries: {dict(counts)} | "
            f"Token budget: {total_tokens_budget:,}/{max_possible:,} "
            f"({total_tokens_budget/max_possible:.0%} of max)"
        )

# Demo: Route a variety of queries
router = ModelRouter()

test_queries = [
    "What is the capital of France?",
    "What is 347 * 23?",
    "How many planets are in our solar system?",
    "If a store has a 25% off sale, and an item costs $60, what is the final price after 8% tax?",
    "Compare the advantages and disadvantages of caching vs compression for agent optimization.",
    "What is the boiling point of water?",
    "If it takes 5 machines 5 minutes to make 5 widgets, then how long would it take 100 machines?",
]

print("Model Router Demo:")
print("=" * 70)
for q in test_queries:
    params = router.get_params(q)
    diff = router.classify_difficulty(q)
    print(f"  [{diff:<8}] max_tokens={params['max_new_tokens']:<4} temp={params['temperature']} | {q[:60]}")

print(f"\n{router.stats()}")

## 6. Circularity Detection — Stop Agents Going in Circles

A common failure mode: the agent repeats the same action over and over, consuming tokens without progress. We detect this and terminate early.

In [ ]:
class CircularityDetector:
    """Detect when an agent is repeating actions and going in circles."""

    def __init__(self, window_size: int = 5, similarity_threshold: float = 0.8):
        """
        Args:
            window_size: Number of recent actions to check for repetition.
            similarity_threshold: Fraction of overlap to consider "circular."
        """
        self.window_size = window_size
        self.similarity_threshold = similarity_threshold
        self.action_history: List[str] = []
        self.circular_detected = False

    def _normalize_action(self, action: str) -> str:
        """Normalize action string for comparison."""
        return re.sub(r'\s+', ' ', action.lower().strip())

    def _similarity(self, a: str, b: str) -> float:
        """Simple word overlap similarity."""
        words_a = set(a.split())
        words_b = set(b.split())
        if not words_a or not words_b:
            return 0.0
        overlap = words_a & words_b
        return len(overlap) / max(len(words_a), len(words_b))

    def record_action(self, action: str) -> bool:
        """Record an action and check for circularity.

        Returns True if circularity is detected.
        """
        normalized = self._normalize_action(action)
        self.action_history.append(normalized)

        if len(self.action_history) < 3:
            return False

        # Check: is the last action very similar to recent ones?
        recent = self.action_history[-self.window_size:]
        if len(recent) >= 3:
            # Count near-duplicates of the latest action
            duplicates = sum(
                1 for prev in recent[:-1]
                if self._similarity(prev, recent[-1]) >= self.similarity_threshold
            )
            if duplicates >= 2:  # Same action 3+ times
                self.circular_detected = True
                return True

        return False

    def reset(self):
        self.action_history = []
        self.circular_detected = False

# Demo: Simulate a circular agent
detector = CircularityDetector()

actions = [
    "Action: calculator, Input: 347 * 23",
    "Action: calculator, Input: 347 * 23",
    "Action: calculator, Input: 347 * 23",  # 3rd repeat → circular!
    "Action: knowledge_base, Input: capital of France",
]

print("Circularity Detection Demo:")
print("-" * 50)
for i, action in enumerate(actions):
    is_circular = detector.record_action(action)
    print(f"  Step {i+1}: {action[:50]}")
    if is_circular:
        print(f"    🔄 CIRCULAR DETECTED — agent is repeating itself!")
        print(f"    → Early termination recommended to save tokens.")
        break

## 7. Putting It All Together — Optimized Agent

Now we combine all optimizations into a single agent wrapper that applies them in order.

In [ ]:
class OptimizedAgent:
    """Agent wrapper that applies all cost optimizations."""

    def __init__(self, tools: Dict, budget_limit: int = 30000):
        self.tools = tools
        self.budget = TokenBudget(max_tokens=budget_limit)
        self.cache = ResponseCache()
        self.compressor = PromptCompressor(max_history_tokens=1500)
        self.router = ModelRouter()
        self.circularity = CircularityDetector()

    def run(self, question: str, max_steps: int = 5) -> Dict:
        """Run the optimized agent on a question."""
        self.circularity.reset()

        tools_desc = "\n".join(
            f"- {name}: {info['description']}" for name, info in self.tools.items()
        )
        system_prompt = f"""You are a helpful assistant with access to these tools:
{tools_desc}

To use a tool, respond with:
Thought: [reasoning]
Action: [tool_name]
Action Input: [input]

When you have the final answer:
Thought: [reasoning]
Final Answer: [answer]

Be concise. Use tools when helpful."""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ]

        tools_used = []
        gen_params = self.router.get_params(question)

        for step in range(max_steps):
            # Check budget
            if self.budget.is_over_budget():
                return {"answer": "Budget exceeded.", "tools_used": tools_used,
                        "steps": step, "optimizations": self._opt_stats()}

            # Compress history
            messages = self.compressor.compress(messages)

            # Try cache
            response, was_cached = generate_cached(messages, self.cache, **gen_params)

            # Track budget
            if not was_cached:
                input_text = " ".join(m["content"] for m in messages)
                self.budget.record_call(input_text, response, label=f"step-{step+1}")

            # Check for circularity
            if self.circularity.record_action(response[:200]):
                return {"answer": "Circular reasoning detected — stopping early.",
                        "tools_used": tools_used, "steps": step + 1,
                        "optimizations": self._opt_stats()}

            # Parse response
            final_match = re.search(r'Final Answer:\s*(.+)', response, re.IGNORECASE)
            if final_match:
                return {"answer": final_match.group(1).strip(),
                        "tools_used": tools_used, "steps": step + 1,
                        "optimizations": self._opt_stats()}

            action_match = re.search(r'Action:\s*(\w+)', response)
            input_match = re.search(r'Action Input:\s*(.+)', response)

            if action_match and input_match:
                tool_name = action_match.group(1).strip()
                tool_input = input_match.group(1).strip()
                if tool_name in self.tools:
                    tools_used.append(tool_name)
                    result = self.tools[tool_name]["fn"](tool_input)
                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content": f"Observation: {result}"})
                else:
                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content":
                        f"Observation: Tool '{tool_name}' not found."})
            else:
                return {"answer": response.strip().split("\n")[0],
                        "tools_used": tools_used, "steps": step + 1,
                        "optimizations": self._opt_stats()}

        return {"answer": "Max steps reached.", "tools_used": tools_used,
                "steps": max_steps, "optimizations": self._opt_stats()}

    def _opt_stats(self) -> Dict:
        return {
            "budget": self.budget.summary(),
            "cache": self.cache.stats(),
            "compression": self.compressor.stats(),
            "routing": self.router.stats(),
        }

print("✓ OptimizedAgent defined")

## 8. Define Tools and Test Queries

In [ ]:
# Tools for our agents
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        allowed = {'abs': abs, 'round': round, 'min': min, 'max': max,
                   'pow': pow, 'sqrt': math.sqrt, 'pi': math.pi, 'e': math.e}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def knowledge_base(query: str) -> str:
    """Look up facts."""
    kb = {
        "capital of france": "Paris is the capital of France.",
        "chemical symbol for gold": "The chemical symbol for gold is Au.",
        "who wrote 1984": "George Orwell wrote '1984'.",
        "boiling point of water": "The boiling point of water is 100°C (212°F).",
        "planets in solar system": "There are 8 planets in our solar system.",
    }
    query_lower = query.lower()
    for key, value in kb.items():
        if key in query_lower or any(w in query_lower for w in key.split()):
            return value
    return f"No info found for: {query}"

TOOLS = {
    "calculator": {"fn": calculator, "description": "Evaluate a math expression."},
    "knowledge_base": {"fn": knowledge_base, "description": "Look up factual information."},
}

# 5 test queries for the comparison experiment
TEST_QUERIES = [
    "What is 347 * 23?",
    "What is the capital of France?",
    "Calculate 15% tip on a $84.50 bill and round to the nearest cent.",
    "What is the boiling point of water in Fahrenheit?",
    "If a rectangle has area 120 and width 8, what is its length?",
]

print(f"✓ {len(TOOLS)} tools and {len(TEST_QUERIES)} test queries ready")

## 9. Comparative Experiment — Measuring Optimization Impact

We run the same 5 tasks through four configurations:

1. **Baseline** — no optimizations
2. **Cached** — response cache enabled (run twice to see cache hits)
3. **Compressed** — prompt compression enabled
4. **Full optimization** — all optimizations enabled

We measure tokens, latency, and answer quality for each.

In [ ]:
def run_baseline_agent(question: str, max_steps: int = 5) -> Dict:
    """Unoptimized baseline agent."""
    tools_desc = "\n".join(f"- {n}: {t['description']}" for n, t in TOOLS.items())
    system_prompt = f"""You are a helpful assistant with tools:
{tools_desc}

Use: Thought/Action/Action Input or Thought/Final Answer format."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    tools_used = []
    total_input_tokens = 0
    total_output_tokens = 0

    for step in range(max_steps):
        input_text = " ".join(m["content"] for m in messages)
        response = generate(messages, max_new_tokens=256, temperature=0.5, do_sample=True)
        total_input_tokens += int(len(input_text.split()) * 1.3)
        total_output_tokens += int(len(response.split()) * 1.3)

        final_match = re.search(r'Final Answer:\s*(.+)', response, re.IGNORECASE)
        if final_match:
            return {"answer": final_match.group(1).strip(), "tools_used": tools_used,
                    "steps": step + 1, "input_tokens": total_input_tokens,
                    "output_tokens": total_output_tokens}

        action_match = re.search(r'Action:\s*(\w+)', response)
        input_match = re.search(r'Action Input:\s*(.+)', response)
        if action_match and input_match:
            tool_name = action_match.group(1).strip()
            tool_input = input_match.group(1).strip()
            if tool_name in TOOLS:
                tools_used.append(tool_name)
                result = TOOLS[tool_name]["fn"](tool_input)
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: {result}"})
            else:
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: Unknown tool."})
        else:
            return {"answer": response.strip().split("\n")[0], "tools_used": tools_used,
                    "steps": step + 1, "input_tokens": total_input_tokens,
                    "output_tokens": total_output_tokens}

    return {"answer": "Max steps.", "tools_used": tools_used, "steps": max_steps,
            "input_tokens": total_input_tokens, "output_tokens": total_output_tokens}

# --- Run experiments ---
print("=" * 70)
print("  OPTIMIZATION COMPARISON EXPERIMENT")
print("=" * 70)

# Experiment 1: Baseline
print("\n--- Baseline (no optimizations) ---")
baseline_results = []
baseline_start = time.time()
for q in TEST_QUERIES:
    start = time.time()
    result = run_baseline_agent(q)
    elapsed = time.time() - start
    result["latency"] = elapsed
    baseline_results.append(result)
    print(f"  [{elapsed:.1f}s] {q[:40]}... → {result['answer'][:40]}")
baseline_total_time = time.time() - baseline_start
baseline_total_tokens = sum(r.get("input_tokens", 0) + r.get("output_tokens", 0) for r in baseline_results)
print(f"  Total: {baseline_total_tokens:,} tokens, {baseline_total_time:.1f}s")

In [ ]:
# Experiment 2: Full optimization
print("\n--- Optimized (all optimizations) ---")
opt_agent = OptimizedAgent(TOOLS, budget_limit=50000)
optimized_results = []
opt_start = time.time()
for q in TEST_QUERIES:
    start = time.time()
    result = opt_agent.run(q)
    elapsed = time.time() - start
    result["latency"] = elapsed
    optimized_results.append(result)
    print(f"  [{elapsed:.1f}s] {q[:40]}... → {result['answer'][:40]}")
opt_total_time = time.time() - opt_start

# Experiment 3: Run optimized again (to show cache hits)
print("\n--- Optimized (2nd run — cache warm) ---")
cached_results = []
cached_start = time.time()
for q in TEST_QUERIES:
    start = time.time()
    result = opt_agent.run(q)
    elapsed = time.time() - start
    result["latency"] = elapsed
    cached_results.append(result)
    print(f"  [{elapsed:.1f}s] {q[:40]}... → {result['answer'][:40]}")
cached_total_time = time.time() - cached_start

# Print optimization stats
print("\n" + "=" * 70)
print("  RESULTS SUMMARY")
print("=" * 70)
print(f"\n  {'Config':<25} {'Tokens':>10} {'Time':>10} {'Savings':>10}")
print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'Baseline':<25} {baseline_total_tokens:>10,} {baseline_total_time:>9.1f}s {'—':>10}")
print(f"  {'Optimized (cold)':<25} {'—':>10} {opt_total_time:>9.1f}s "
      f"{max(0,(baseline_total_time-opt_total_time)/baseline_total_time*100):>9.0f}%")
print(f"  {'Optimized (warm cache)':<25} {'—':>10} {cached_total_time:>9.1f}s "
      f"{max(0,(baseline_total_time-cached_total_time)/baseline_total_time*100):>9.0f}%")

print(f"\nOptimization details:")
stats = opt_agent._opt_stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

## 10. Key Takeaways

### What We Built

| Component | Savings | How It Works |
|-----------|---------|-------------|
| **TokenBudget** | Awareness | Track consumption, alert at thresholds, stop over-budget |
| **ResponseCache** | Up to 100% per repeat | Hash prompts, reuse identical responses |
| **PromptCompressor** | 30–60% per call | Summarize history, keep only recent context |
| **ModelRouter** | 50–75% per simple task | Use short/focused generation for easy tasks |
| **CircularityDetector** | Variable | Catch loops early, stop wasting tokens |

### Optimization Principles

1. **Measure first** — you can't optimize what you can't measure (TokenBudget)
2. **Don't repeat work** — cache responses for identical prompts
3. **Compress context** — agents accumulate history; summarize it
4. **Right-size generation** — not every task needs 512 tokens of output
5. **Fail fast** — detect loops and stop early rather than burning budget

### Real-World Considerations

- **Cache invalidation**: In production, add TTL (time-to-live) to cache entries
- **Compression quality**: Better summarization = better compression without information loss
- **Routing accuracy**: Misclassifying hard tasks as easy hurts quality; when in doubt, route up
- **Combined savings**: Optimizations compound — caching + compression + routing together save more than any one alone

### What's Next

In the next notebook, we build **error handling and resilience patterns** — retry with feedback, fallback chains, and circuit breakers.